
### DBpedia Querying & Filtering for Successful Queries
Iterates through unique queries, sends to DBpedia, saves queries with non-empty results


In [ ]:
import pandas as pd
import requests
import json
import time
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import os

DBPEDIA_ENDPOINT = "https://dbpedia.data.dice-research.org/sparql"
MIN_RESULTS_PER_QUERY = 3  # Optional: keep queries with at least N results
OUTPUT_FILE = "queries_with_answers.json"
SAVE_EVERY = 100  # intermittent save

# 1. Load unique SELECT queries (try common column names)
df = pd.read_csv("unique_select_queries.csv")
if "query" in df.columns:
    query_col = "query"
elif "text" in df.columns:
    query_col = "text"
elif "sparql_query" in df.columns:
    query_col = "sparql_query"
else:
    raise ValueError(f"CSV missing expected column (query/text/sparql_query). Columns: {list(df.columns)}")


In [ ]:
# 2. Prepare requests.Session with retries
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=(429, 500, 502, 503, 504))
session.mount("https://", HTTPAdapter(max_retries=retries))

HEADERS = {"Accept": "application/sparql-results+json"}

def query_dbpedia(sparql_query, timeout=15):
    try:
        resp = session.get(DBPEDIA_ENDPOINT,
                           params={"query": sparql_query},
                           headers=HEADERS,
                           timeout=timeout)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        return {"error": str(e)}
    

In [ ]:

# 3. Run queries and collect successful ones
successful_queries = []
start_idx = 0

for i, row in tqdm(df.iterrows(), total=len(df), desc="Querying DBpedia"):
    sparql = str(row[query_col]).strip()
    if not sparql:
        continue
    res = query_dbpedia(sparql)
    # handle network/errors
    if isinstance(res, dict) and "error" in res:
        # skip or log, here we skip
        continue
    bindings = res.get("results", {}).get("bindings", [])
    if len(bindings) >= MIN_RESULTS_PER_QUERY:
        successful_queries.append({
            "sparql_query": sparql,
            "results_count": len(bindings),
            "results": bindings
        })
    # polite delay to avoid rate-limiting
    time.sleep(0.1)

    # intermittent save
    if len(successful_queries) % SAVE_EVERY == 0 and successful_queries:
        tmp_path = OUTPUT_FILE + ".partial"
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(successful_queries, f, ensure_ascii=False, indent=2)



In [ ]:
# 4. Final save
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(successful_queries, f, ensure_ascii=False, indent=2)

print(f"Saved {len(successful_queries)} queries with answers to {OUTPUT_FILE}")